In [6]:
import numpy as np

# Exercise 1a: Linear Algebra Foundations

**3D Computer Vision — Summer Semester 2026**

**Prof. Dr. Daniel Cremers, TU Munich**

---

Almost every algorithm in 3D computer vision reduces to linear algebra at its core. Triangulating a 3D point from two camera views means solving a linear system. Estimating a camera's pose means decomposing a matrix with the SVD. Fitting a model to noisy measurements means computing a pseudo-inverse.

In this notebook you will implement four building blocks that the rest of the course relies on:

| Part | Topic | Key idea |
|------|-------|----------|
| I | Gaussian elimination | Inverting a matrix by elementary row operations |
| II | Null space | Finding vectors that a matrix maps to zero (via SVD) |
| III | Pseudo-inverse | Solving over- and under-determined systems (via SVD) |
| IV | Subspace intersection | Using the null space to find where two subspaces meet |

## Part I: Gaussian Elimination

Given a square matrix $A \in \mathbb{R}^{n \times n}$, we want to compute $A^{-1}$ using only three elementary row operations:

| Operation | Notation | Effect on $[A \mid I]$ |
|-----------|----------|------------------------|
| Swap | `("S", i, j)` | Exchange rows $i$ and $j$ |
| Scale | `("M", i, s)` | Multiply row $i$ by scalar $s \neq 0$ |
| Add | `("A", i, j, s)` | Replace row $i$ with row$_i$ + $s \cdot$ row$_j$ |

**Algorithm:** Form the augmented matrix $[A \mid I]$. Apply row operations to reduce the left half to $I$. The right half becomes $A^{-1}$.

Use **partial pivoting** for numerical stability: before eliminating column $j$, swap the current row with the row below it that has the largest absolute value in column $j$.

**Implement** `gaussian_elimination_inverse` below. Track each operation in the `ops` list. Append `"SOLUTION"` when done, or `"DEGENERATE"` if a zero pivot is encountered (the matrix is singular).

In [7]:
def gaussian_elimination_inverse(A):
    """Compute the inverse of A using Gaussian elimination."""
    n = A.shape[0]
    aug = np.hstack([A.astype(float), np.eye(n)])
    ops = []

    for i in range(n):
        j = i + np.argmax(np.abs(aug[i:, i]))

        if np.isclose(aug[j, i], 0):
            ops.append("DEGENERATE")
            return ops, None

        if j != i:
            aug[[i, j]] = aug[[j, i]]
            ops.append(("S", i, j))

        pivot = aug[i, i]
        aug[i] = aug[i] / pivot
        ops.append(("M", i, 1 / pivot))

        for j in range(n):
            if j != i:
                factor = aug[j, i]
                aug[j] = aug[j] - factor * aug[i]
                ops.append(("A", j, i, -factor))

    A_inv = aug[:, n:]

    return ops, A_inv

In [8]:
def replay_operations(A, ops):
    """Replay elementary row operations on [A | I] and return the result."""
    n = A.shape[0]
    aug = np.hstack([A.astype(float), np.eye(n)])
    for op in ops:
        if op in ("SOLUTION", "DEGENERATE"):
            break
        if op[0] == "S":
            _, i, j = op
            aug[[i, j]] = aug[[j, i]]
        elif op[0] == "M":
            _, i, s = op
            aug[i] *= s
        elif op[0] == "A":
            _, i, j, s = op
            aug[i] += s * aug[j]
    return aug

In [9]:
# ── Tests for Part I ──
np.random.seed(42)
_A = np.random.randn(4, 4)
_ops, _A_inv = gaussian_elimination_inverse(_A)
assert np.allclose(_A @ _A_inv, np.eye(4), atol=1e-10), "A @ A_inv != I"
_aug = replay_operations(_A, _ops)
assert np.allclose(_aug[:, :4], np.eye(4), atol=1e-10), "Ops don't produce I on left"
assert np.allclose(_aug[:, 4:], _A_inv, atol=1e-10), "Ops don't produce A_inv on right"

_A_sing = np.array([[1, 2], [2, 4]], dtype=float)
_ops_s, _inv_s = gaussian_elimination_inverse(_A_sing)
assert _inv_s is None and _ops_s[-1] == "DEGENERATE", "Singular not detected"

print(f"Part I: ALL PASS ({len(_ops)-1} operations)")

Part I: ALL PASS (17 operations)


## Part II: Null Space Computation

The **null space** (or kernel) of $D \in \mathbb{R}^{m \times n}$ is the set of all vectors that $D$ maps to zero:

$$\ker(D) = \{ \mathbf{x} \in \mathbb{R}^n \mid D\mathbf{x} = \mathbf{0} \}$$

The null space tells us about the **ambiguities** in a linear system — directions along which we have no information. This will show up repeatedly:

- In the **8-point algorithm** (Lecture 5), the essential matrix is the null vector of a constraint matrix.
- In **triangulation** (Lecture 6), the 3D point is the null vector of stacked projection constraints.

**Approach:** Compute the SVD $D = U \Sigma V^\top$. The columns of $V$ corresponding to zero singular values form an orthonormal basis for $\ker(D)$.

**Implement** `null_space` below.

In [10]:
def null_space(D, tol=1e-10):
    """Orthonormal basis for the null space of D."""
    U, S, Vt = np.linalg.svd(D)
    r = np.sum(S > tol)
    N = Vt[r:].T
    return N

In [11]:
# ── Tests for Part II ──
_D = np.array([[1, 2, 3], [4, 5, 6], [7, 8, 9]], dtype=float)
_N = null_space(_D)
assert _N.shape[1] == 1, f"Expected null space dim 1, got {_N.shape[1]}"
assert np.max(np.abs(_D @ _N)) < 1e-10, "D @ N != 0"
assert null_space(np.eye(3)).shape[1] == 0, "Identity should have trivial null space"
print("Part II: ALL PASS")

Part II: ALL PASS


## Part III: Moore-Penrose Pseudo-Inverse

Given a (possibly rectangular, possibly singular) matrix $D \in \mathbb{R}^{m \times n}$, we want to solve $D\mathbf{x} = \mathbf{b}$ "as well as possible." The SVD gives a principled answer. Write $D = U \Sigma V^\top$ and define:

$$D^+ = V \Sigma^+ U^\top, \qquad \text{where } \Sigma^+_{ii} = \begin{cases} 1/\sigma_i & \sigma_i > 0 \\ 0 & \sigma_i = 0 \end{cases}$$

Then $\hat{\mathbf{x}} = D^+ \mathbf{b}$ is:

- The **least-squares solution** if $m > n$ (overdetermined — more equations than unknowns).
- The **minimum-norm solution** if $m < n$ (underdetermined — infinitely many solutions).

**Implement** `pseudo_inverse` below.

In [20]:
def pseudo_inverse(D, tol=1e-10):
    """Moore-Penrose pseudo-inverse via SVD."""
    U, S, Vt = np.linalg.svd(D, full_matrices=False)
    return Vt.T @ np.diag([1/s if s > tol else 0 for s in S]) @ U.T
def solve_linear_svd(D, b, tol=1e-10):
    """Solve Dx = b via pseudo-inverse."""
    D_pinv = pseudo_inverse(D, tol)
    return D_pinv @ b, D_pinv

In [21]:
# ── Tests for Part III ──
np.random.seed(7)
_D = np.random.randn(10, 3)
_x_true = np.array([1.0, 2.0, 3.0])
_b = _D @ _x_true + 0.01 * np.random.randn(10)
_x_hat, _D_pinv = solve_linear_svd(_D, _b)
assert np.allclose(_D_pinv, np.linalg.pinv(_D), atol=1e-10), "Pinv doesn't match numpy"
print(f"  Overdetermined: ||x_hat - x_true|| = {np.linalg.norm(_x_hat - _x_true):.2e}")

_D2 = np.array([[1, 2, 3], [4, 5, 6]], dtype=float)
_b2 = np.array([14.0, 32.0])
_x2, _ = solve_linear_svd(_D2, _b2)
_N2 = null_space(_D2)
assert np.allclose(_D2 @ _x2, _b2, atol=1e-10), "Dx != b"
assert np.allclose(_D2 @ (_x2 + 5.0 * _N2.flatten()), _b2, atol=1e-10), "Null space broken"
print(f"  Underdetermined: ||x_hat|| = {np.linalg.norm(_x2):.4f} (minimum-norm)")
print("Part III: ALL PASS")

  Overdetermined: ||x_hat - x_true|| = 1.14e-03
  Underdetermined: ||x_hat|| = 3.7417 (minimum-norm)
Part III: ALL PASS


## Part IV: Subspace Intersection

Two people each know a set of landmark positions in $\mathbb{R}^m$. Person A can reach any point in $\mathrm{span}(P_A)$, person B any point in $\mathrm{span}(P_B)$. Where can they meet?

A point $\mathbf{x}$ lies in both subspaces iff $\mathbf{x} = P_A \boldsymbol{\alpha} = P_B \boldsymbol{\beta}$. Rearranging:

$$\begin{pmatrix} P_A & -P_B \end{pmatrix} \begin{pmatrix} \boldsymbol{\alpha} \\ \boldsymbol{\beta} \end{pmatrix} = \mathbf{0}$$

So we need the **null space** of $[P_A \mid -P_B]$, then reconstruct intersection points as $P_A \boldsymbol{\alpha}$.

**Implement** `meeting_point` using your `null_space` from Part II.

In [25]:
def meeting_point(pts_a, pts_b):
    """Find the intersection of span(pts_a) and span(pts_b)."""
    M = np.hstack([pts_a, -pts_b])
    N = null_space(M)

    k = pts_a.shape[1]
    alpha = N[:k]
    X = pts_a @ alpha

    return X

In [24]:
# ── Tests for Part IV ──
_P_A = np.array([[1, 0], [0, 1], [0, 0]], dtype=float)
_P_B = np.array([[1, 0], [0, 0], [0, 1]], dtype=float)
_inter = meeting_point(_P_A, _P_B)
_dir = _inter[:, 0] / np.linalg.norm(_inter[:, 0])
assert _inter.shape[1] == 1, f"Expected dim 1, got {_inter.shape[1]}"
assert np.allclose(np.abs(_dir), [1, 0, 0], atol=1e-10), "Not along x-axis"
print("Part IV: ALL PASS")

Part IV: ALL PASS
